# carGO PH — IoT Simulation
**Course:** MO-IT148 — Application Development and Emerging Technologies  
**Group:** NodeBlk

**Week:** 2 — IoT Data Simulation  
**Description:** Simulates raw sensor data (GPS, RFID, Temperature) for a smart logistics tracking system.  
Generates the Shipment Registry first, then attaches 3 independent sensor scripts to it.  
Only temp-regulated goods categories receive Temperature readings. Exports to CSV for blockchain processing.

> **WEEK 9 | v2 changes vs v1:** NUM_SHIPMENTS 30→50, READINGS_PER_SHIPMENT 5→8, RFID scans 2–4→3–6,  
> RFID flag rate 15%→25%, temp breach rate 10%→20%, shipment registry now includes  
> `scheduled_delivery_dt`, `actual_delivery_dt`, `shipment_status`, `delay_reason`.

## 1. Setup
Imports and global constants. All counts and bounds live here — generators read from these, never hard-code their own.

In [ ]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

random.seed(42)   # reproducible output — remove for fully random each run

# ── Simulation scale ──────────────────────────────────────────────────────
NUM_SHIPMENTS              = 50    # was 30 — more shipments = richer origin/category charts
READINGS_PER_SHIPMENT      = 8     # was 5  — denser sensor timeline & temp trend
MAX_DEPARTURE_OFFSET_HOURS = 72    # was 48 — 3-day window spreads heatmap better
START_TIMESTAMP            = datetime(2026, 5, 3, 6, 0, 0)

# ── Goods categories ──────────────────────────────────────────────────────
TEMP_REGULATED_CATEGORIES = [
    "Deep Freeze", "Frozen", "Chill/Refrigerated", "Pharma", "Cool-Chain"
]
NON_TEMP_CATEGORIES = ["Dry Goods", "Electronics", "Clothing", "Industrial"]
GOODS_CATEGORIES    = TEMP_REGULATED_CATEGORIES + NON_TEMP_CATEGORIES

# Temperature safe ranges (°C) — matches retrieval notebook exactly
TEMP_RANGES = {
    "Deep Freeze":        (-30.0, -28.0),
    "Frozen":             (-20.0, -16.0),
    "Chill/Refrigerated": (2.0,    4.0),
    "Pharma":             (2.0,    8.0),
    "Cool-Chain":         (12.0,  14.0),
}

# ── City data ─────────────────────────────────────────────────────────────
cities_df   = pd.read_csv("../data/ph.csv")
cities_df   = cities_df[["city", "lat", "lng"]].dropna().drop_duplicates(subset="city")
CITIES      = cities_df["city"].tolist()
CITY_COORDS = dict(zip(cities_df["city"], zip(cities_df["lat"], cities_df["lng"])))

print(f"Cities loaded: {len(CITIES)}")
print(f"Simulation: {NUM_SHIPMENTS} shipments × {READINGS_PER_SHIPMENT} readings/sensor")


Cities loaded: 57
Simulation: 50 shipments × 8 readings/sensor


## 2. Shipment Registry
Generated first — all sensor scripts reference this. One row per shipment.  
**New in v2:** adds `scheduled_delivery_dt`, `actual_delivery_dt`, `shipment_status`, `delay_reason`.

| Column | Format | Description |
|--------|--------|-------------|
| `rfid_tag` | `RFID-001`…`RFID-050` | Unique shipment identifier (foreign key for all sensor tables) |
| `goods_category` | e.g. `Frozen` | Determines whether a temperature sensor is attached |
| `origin` / `destination` | Philippine city name | Used for GPS coordinate interpolation |
| `package_count` | 1–50 | Number of packages in the shipment |
| `vehicle_id` / `driver_id` | `VH-001`…`VH-050` | Assigned vehicle and driver |
| `scheduled_delivery_dt` | datetime | Expected arrival (departure + 12–72 h transit) |
| `actual_delivery_dt` | datetime | Real arrival — same as scheduled if on-time, later if delayed |
| `shipment_status` | `Delivered` / `Delayed` | 20% chance of delay |
| `delay_reason` | string or `None` | Cause of delay: Route change, Weather, RFID flag, Temp breach, Traffic |

In [2]:
registry_rows = []

for i in range(NUM_SHIPMENTS):
    rfid_tag = f"RFID-{i+1:03d}"
    origin   = random.choice(CITIES)
    dest     = random.choice([c for c in CITIES if c != origin])
    dep_offset = random.randint(0, MAX_DEPARTURE_OFFSET_HOURS)

    # Delivery timing
    departure_dt  = START_TIMESTAMP + timedelta(hours=dep_offset)
    transit_hours = random.randint(12, 72)
    scheduled_dt  = departure_dt + timedelta(hours=transit_hours)

    # Delay logic — 20% of shipments are delayed
    is_delayed    = random.random() < 0.20
    delay_hours   = random.randint(2, 12) if is_delayed else 0
    actual_dt     = scheduled_dt + timedelta(hours=delay_hours)
    status        = "Delayed" if is_delayed else "Delivered"
    delay_reason  = random.choice(
        ["Route change", "Weather", "RFID flag", "Temp breach", "Traffic"]
    ) if is_delayed else "None"

    registry_rows.append({
        "rfid_tag":              rfid_tag,
        "goods_category":        random.choice(GOODS_CATEGORIES),
        "origin":                origin,
        "destination":           dest,
        "package_count":         random.randint(1, 50),
        "vehicle_id":            f"VH-{i+1:03d}",
        "driver_id":             f"DR-{i+1:03d}",
        "scheduled_delivery_dt": scheduled_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "actual_delivery_dt":    actual_dt.strftime("%Y-%m-%d %H:%M:%S"),
        "shipment_status":       status,
        "delay_reason":          delay_reason,
        "departure_offset":      dep_offset,   # dropped before CSV export
    })

shipment_registry_df = pd.DataFrame(registry_rows)

print(f"Shipment Registry: {len(shipment_registry_df)} rows")
print(f"Status breakdown: {shipment_registry_df['shipment_status'].value_counts().to_dict()}")
print(f"Delay reasons: {shipment_registry_df[shipment_registry_df['delay_reason']!='None']['delay_reason'].value_counts().to_dict()}")
display(shipment_registry_df.drop(columns=['departure_offset']).head())


Shipment Registry: 50 rows
Status breakdown: {'Delivered': 41, 'Delayed': 9}
Delay reasons: {'RFID flag': 3, 'Traffic': 2, 'Weather': 2, 'Temp breach': 1, 'Route change': 1}


,rfid_tag,goods_category,origin,destination,package_count,vehicle_id,driver_id,scheduled_delivery_dt,actual_delivery_dt,shipment_status,delay_reason
0,RFID-001,Pharma,Tatalon,Pasig City,9,VH-001,DR-001,2026-05-05 20:00:00,2026-05-05 20:00:00,Delivered,None
1,RFID-002,Deep Freeze,Bayanan,Taguig City,2,VH-002,DR-002,2026-05-06 20:00:00,2026-05-06 20:00:00,Delivered,None
2,RFID-003,Industrial,Canagatan,Mandaluyong City,13,VH-003,DR-003,2026-05-06 07:00:00,2026-05-06 07:00:00,Delivered,None
3,RFID-004,Cool-Chain,Central Signal Village,Don Bosco,1,VH-004,DR-004,2026-05-07 17:00:00,2026-05-07 17:00:00,Delivered,None
4,RFID-005,Cool-Chain,Karuhatan,Quiapo,10,VH-005,DR-005,2026-05-06 10:00:00,2026-05-06 10:00:00,Delivered,None


## 3. GPS Sensor Readings
Every shipment gets GPS regardless of goods category.  
Coordinates interpolate from origin → destination across `READINGS_PER_SHIPMENT` pings.  
**Output:** `gps_readings.csv` — 50 × 8 = **400 rows**

In [3]:
gps_rows = []

for _, shipment in shipment_registry_df.iterrows():
    shipment_start = START_TIMESTAMP + timedelta(hours=shipment["departure_offset"])
    origin_coords  = CITY_COORDS[shipment["origin"]]
    dest_coords    = CITY_COORDS[shipment["destination"]]

    for j in range(READINGS_PER_SHIPMENT):
        fraction  = j / (READINGS_PER_SHIPMENT - 1)
        lat = round(origin_coords[0] + fraction * (dest_coords[0] - origin_coords[0]) + random.uniform(-0.05, 0.05), 6)
        lng = round(origin_coords[1] + fraction * (dest_coords[1] - origin_coords[1]) + random.uniform(-0.05, 0.05), 6)
        timestamp = shipment_start + timedelta(hours=j * 2)

        gps_rows.append({
            "reading_id": f"RDG-GPS-{len(gps_rows)+1:04d}",
            "rfid_tag":   shipment["rfid_tag"],
            "device_id":  f"GPS{random.randint(100,999)}",
            "data_type":  "GPS",
            "data_value": f"{lat},{lng}",
            "timestamp":  timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        })

gps_df          = pd.DataFrame(gps_rows)
gps_df["gps_lat"] = gps_df["data_value"].apply(lambda x: float(x.split(",")[0]))
gps_df["gps_lng"] = gps_df["data_value"].apply(lambda x: float(x.split(",")[1]))

print(f"GPS Readings: {len(gps_df)} rows")
display(gps_df.head())


GPS Readings: 400 rows


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp,gps_lat,gps_lng
0,RDG-GPS-0001,RFID-001,GPS926,GPS,"14.644554,121.026468",2026-05-03 09:00:00,14.644554,121.026468
1,RDG-GPS-0002,RFID-001,GPS936,GPS,"14.630498,120.974825",2026-05-03 11:00:00,14.630498,120.974825
2,RDG-GPS-0003,RFID-001,GPS206,GPS,"14.61146,121.075961",2026-05-03 13:00:00,14.611460,121.075961
3,RDG-GPS-0004,RFID-001,GPS218,GPS,"14.640793,121.004958",2026-05-03 15:00:00,14.640793,121.004958
4,RDG-GPS-0005,RFID-001,GPS259,GPS,"14.626771,121.074513",2026-05-03 17:00:00,14.626771,121.074513


## 4. RFID Sensor Readings
Every shipment gets RFID scans at checkpoints.  
**v2 changes:** scans per shipment 2–4 → **3–6**, flag rate 15% → **25%** for heatmap density.  
**Output:** `rfid_readings.csv` — ~50 × 4.5 avg = **~225 rows**, ~56 flagged

In [4]:
rfid_rows = []

for _, shipment in shipment_registry_df.iterrows():
    shipment_start = START_TIMESTAMP + timedelta(hours=shipment["departure_offset"])
    num_scans      = random.randint(3, 6)   # was 2-4

    for j in range(num_scans):
        timestamp   = shipment_start + timedelta(hours=j * 4 + random.randint(0, 3))
        rfid_status = "VERIFIED" if random.random() > 0.25 else "FLAGGED"  # was 0.15

        rfid_rows.append({
            "reading_id": f"RDG-RFD-{len(rfid_rows)+1:04d}",
            "rfid_tag":   shipment["rfid_tag"],
            "device_id":  f"RFD{random.randint(100,999)}",
            "data_type":  "RFID",
            "data_value": rfid_status,
            "timestamp":  timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        })

rfid_df = pd.DataFrame(rfid_rows)
flagged_count = (rfid_df["data_value"] == "FLAGGED").sum()
print(f"RFID Readings: {len(rfid_df)} rows | Flagged: {flagged_count} ({flagged_count/len(rfid_df)*100:.1f}%)")
display(rfid_df.head())


RFID Readings: 226 rows | Flagged: 52 (23.0%)


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp
0,RDG-RFD-0001,RFID-001,RFD771,RFID,VERIFIED,2026-05-03 11:00:00
1,RDG-RFD-0002,RFID-001,RFD825,RFID,FLAGGED,2026-05-03 16:00:00
2,RDG-RFD-0003,RFID-001,RFD485,RFID,VERIFIED,2026-05-03 17:00:00
3,RDG-RFD-0004,RFID-001,RFD786,RFID,VERIFIED,2026-05-03 23:00:00
4,RDG-RFD-0005,RFID-002,RFD118,RFID,VERIFIED,2026-05-06 06:00:00


## 5. Temperature Sensor Readings
Only temp-regulated categories receive a temperature sensor.  
**v2 changes:** breach rate 10% → **20%** for richer breach analysis.  
**Output:** `temperature_readings.csv` — ~30 temp-reg shipments × 8 = **~240 rows**, ~48 breaches

In [5]:
temp_rows = []

for _, shipment in shipment_registry_df.iterrows():
    shipment_start = START_TIMESTAMP + timedelta(hours=shipment["departure_offset"])
    category       = shipment["goods_category"]

    if category not in TEMP_REGULATED_CATEGORIES:
        continue

    low, high = TEMP_RANGES[category]

    for j in range(READINGS_PER_SHIPMENT):
        timestamp = shipment_start + timedelta(hours=j * 2)

        if random.random() < 0.20:   # was 0.10 — breach rate doubled
            temperature = round(random.uniform(high, high + 3.0), 1)
        else:
            temperature = round(random.uniform(low, high), 1)

        temp_rows.append({
            "reading_id": f"RDG-TMP-{len(temp_rows)+1:04d}",
            "rfid_tag":   shipment["rfid_tag"],
            "device_id":  f"TMP{random.randint(100,999)}",
            "data_type":  "Temperature",
            "data_value": str(temperature),
            "timestamp":  timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        })

temp_df = pd.DataFrame(temp_rows)
print(f"Temperature Readings: {len(temp_df)} rows")
print(f"Skipped non-temp categories: {NON_TEMP_CATEGORIES}")
display(temp_df.head())


Temperature Readings: 240 rows
Skipped non-temp categories: ['Dry Goods', 'Electronics', 'Clothing', 'Industrial']


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp
0,RDG-TMP-0001,RFID-001,TMP649,Temperature,3.2,2026-05-03 09:00:00
1,RDG-TMP-0002,RFID-001,TMP632,Temperature,9.0,2026-05-03 11:00:00
2,RDG-TMP-0003,RFID-001,TMP834,Temperature,6.1,2026-05-03 13:00:00
3,RDG-TMP-0004,RFID-001,TMP622,Temperature,2.5,2026-05-03 15:00:00
4,RDG-TMP-0005,RFID-001,TMP235,Temperature,2.3,2026-05-03 17:00:00


## 6. Combine All Sensors → `5-iot_data_v2.csv`
Stacks GPS, RFID, and Temperature into one unified feed sorted by timestamp.  
This is the only file consumed by the blockchain bulk-write loop.

>Note: reading_id order ≠ timestamp order — this is intentional, not a bug.
reading_id is assigned by loop position in Section 3–5 (i.e., shipment registry row order: RFID-001 always gets the lowest IDs, RFID-002 the next block, etc.). timestamp is derived from each shipment's departure_offset, a random value (0–72 hrs) set independently in Section 2. These two are unrelated, so a low-numbered reading (e.g. RDG-GPS-0001, RFID-001) can have a later timestamp than a high-numbered one (e.g. RDG-GPS-0153, RFID-020) if RFID-001 simply rolled a later departure offset.
>
>This mirrors real fleet behavior: shipment IDs reflect booking/registration order, while timestamps reflect actual departure order — and dispatch order rarely matches booking order once routing, vehicle availability, and scheduling are factored in. 5-iot_data_v2.csv is intentionally sorted by timestamp (not reading_id) since downstream consumers — the blockchain bulk-write loop and dashboards — care about chronological event order, not generation order.

In [6]:
iot_data_df = pd.concat([gps_df, rfid_df, temp_df], ignore_index=True)
iot_data_df = iot_data_df.sort_values("timestamp").reset_index(drop=True)

iot_data_df["temperature_c"] = iot_data_df.apply(
    lambda r: float(r["data_value"]) if r["data_type"] == "Temperature" else None, axis=1
)

print(f"Combined IoT Data: {len(iot_data_df)} total rows")
print(iot_data_df["data_type"].value_counts().to_dict())
display(iot_data_df.head(10))


Combined IoT Data: 866 total rows
{'GPS': 400, 'Temperature': 240, 'RFID': 226}


,reading_id,rfid_tag,device_id,data_type,data_value,timestamp,gps_lat,gps_lng,temperature_c
0,RDG-GPS-0153,RFID-020,GPS255,GPS,"14.667702,120.978645",2026-05-03 07:00:00,14.667702,120.978645,NaN
1,RDG-RFD-0082,RFID-020,RFD479,RFID,VERIFIED,2026-05-03 08:00:00,NaN,NaN,NaN
2,RDG-GPS-0001,RFID-001,GPS926,GPS,"14.644554,121.026468",2026-05-03 09:00:00,14.644554,121.026468,NaN
3,RDG-GPS-0154,RFID-020,GPS411,GPS,"14.597148,120.976441",2026-05-03 09:00:00,14.597148,120.976441,NaN
4,RDG-TMP-0001,RFID-001,TMP649,Temperature,3.2,2026-05-03 09:00:00,NaN,NaN,3.2
5,RDG-GPS-0155,RFID-020,GPS671,GPS,"14.562975,120.955133",2026-05-03 11:00:00,14.562975,120.955133,NaN
6,RDG-TMP-0033,RFID-007,TMP935,Temperature,-18.8,2026-05-03 11:00:00,NaN,NaN,-18.8
7,RDG-TMP-0002,RFID-001,TMP632,Temperature,9.0,2026-05-03 11:00:00,NaN,NaN,9.0
8,RDG-GPS-0049,RFID-007,GPS661,GPS,"14.671551,120.940095",2026-05-03 11:00:00,14.671551,120.940095,NaN
9,RDG-GPS-0002,RFID-001,GPS936,GPS,"14.630498,120.974825",2026-05-03 11:00:00,14.630498,120.974825,NaN


## 7. Export

In [8]:
shipment_registry_df.drop(columns=["departure_offset"]).to_csv("../data/1-shipment_registry_v2.csv", index=False)
gps_df.to_csv("../data/2-gps_readings_v2.csv",              index=False)
rfid_df.to_csv("../data/3-rfid_readings_v2.csv",            index=False)
temp_df.to_csv("../data/4-temperature_readings_v2.csv",     index=False)
iot_data_df.to_csv("../data/5-iot_data_v2.csv",             index=False)

temp_count     = shipment_registry_df[shipment_registry_df["goods_category"].isin(TEMP_REGULATED_CATEGORIES)].shape[0]
non_temp_count = NUM_SHIPMENTS - temp_count

print(f"""
Simulation Summary
------------------
Total shipments:             {NUM_SHIPMENTS}
  Temp-regulated:            {temp_count}
  Non-temp:                  {non_temp_count}

GPS readings:                {len(gps_df)}
RFID readings:               {len(rfid_df)}  (flagged: {(rfid_df['data_value']=='FLAGGED').sum()})
Temperature readings:        {len(temp_df)}
Total iot_data.csv rows:     {len(iot_data_df)}

Files written:
  → 1-shipment_registry_v2.csv   ({len(shipment_registry_df)} rows, 11 columns)
  → 2-gps_readings_v2.csv
  → 3-rfid_readings_v2.csv
  → 4-temperature_readings_v2.csv
  → 5-iot_data_v2.csv            (→ goes to blockchain)
""")



Simulation Summary
------------------
Total shipments:             50
  Temp-regulated:            30
  Non-temp:                  20

GPS readings:                400
RFID readings:               226  (flagged: 52)
Temperature readings:        240
Total iot_data.csv rows:     866

Files written:
  → 1-shipment_registry_v2.csv   (50 rows, 11 columns)
  → 2-gps_readings_v2.csv
  → 3-rfid_readings_v2.csv
  → 4-temperature_readings_v2.csv
  → 5-iot_data_v2.csv            (→ goes to blockchain)

